In [ ]:
import os
import cv2
import pandas as pd
import numpy as np
import torch
from facenet_pytorch import MTCNN
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import shutil

In [ ]:
METADATA_PATH = "/content/drive/MyDrive/anti_spoofing_dataset/dataset/metadata.csv"
SOURCE_ROOT = "/content/drive/MyDrive/anti_spoofing_dataset/dataset"
OUTPUT_DIR = "/content/drive/MyDrive/dataset_antispoofing_processed_v1"

In [ ]:
class LivenessDatasetBuilder:
    def __init__(self, metadata_df, source_video_dir, output_dataset_dir):
        """
        Classe unificata per la creazione del dataset di Liveness Detection.
        Aggiornata per usare MTCNN (facenet-pytorch) per il face cropping.
        """
        self.metadata = metadata_df.copy()
        self.source_root = source_video_dir
        self.output_dir = output_dataset_dir
        self.splits = ['train', 'val', 'test']

        # --- MODIFICA 1: Inizializzazione MTCNN ---
        self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        print(f"MTCNN running on device: {self.device}")

        # margin=20 aggiunge un po' di contesto attorno al viso (utile per liveness)
        # select_largest=True evita di prendere volti di sfondo
        self.mtcnn = MTCNN(
            device=self.device,
            select_largest=True,
            margin=20,
            post_process=False # Vogliamo solo le coordinate, non il tensor normalizzato
        )

    def _get_real_video_path(self, kaggle_path):
        """
        Trasforma il path originale dei metadata nel nome file appiattito.
        """
        flattened_name = kaggle_path.replace("/", "-")
        full_path = os.path.join(self.source_root, flattened_name)

        if not os.path.exists(full_path):
            base_name_no_ext = os.path.splitext(flattened_name)[0]
            alternatives = ['.MOV', '.mp4', '.avi', '.mkv']
            for ext in alternatives:
                attempt_path = os.path.join(self.source_root, base_name_no_ext + ext)
                if os.path.exists(attempt_path):
                    return attempt_path
        return full_path

    def create_splits(self, val_size=0.2, test_size=0.1, random_state=42):
        """
        Divide i dati basandosi sui VIDEO univoci per evitare Data Leakage.
        """
        print("Calcolo degli split Train/Val/Test sui video univoci...")
        unique_videos = self.metadata['Video_Path'].unique()

        train_videos, temp_videos = train_test_split(
            unique_videos, test_size=(val_size + test_size), random_state=random_state
        )

        relative_test_size = test_size / (val_size + test_size)
        val_videos, test_videos = train_test_split(
            temp_videos, test_size=relative_test_size, random_state=random_state
        )

        split_map = {}
        for v in train_videos: split_map[v] = 'train'
        for v in val_videos: split_map[v] = 'val'
        for v in test_videos: split_map[v] = 'test'

        self.metadata['split'] = self.metadata['Video_Path'].map(split_map)
        print(f"Split completato. Train: {len(train_videos)} | Val: {len(val_videos)} | Test: {len(test_videos)}")
        return self.metadata

    def extract_and_create(self, frames_per_video=5, crop_faces=True, target_size=(224, 224)):
        """
        Processa il dataset e salva le immagini su disco usando MTCNN per il crop.
        """
        if 'split' not in self.metadata.columns:
            raise ValueError("Errore: Esegui prima .create_splits()!")

        for split in self.splits:
            for label in ['live', 'spoof']:
                os.makedirs(os.path.join(self.output_dir, split, label), exist_ok=True)

        print(f"Inizio estrazione dataset in: {self.output_dir}")
        counters = {'processed': 0, 'missing': 0, 'errors': 0, 'no_faces': 0}

        for idx, row in tqdm(self.metadata.iterrows(), total=len(self.metadata)):
            original_path = row['Video_Path']
            real_path = self._get_real_video_path(original_path)
            label = row['LiveOrSpoof']
            split = row['split']

            if not os.path.exists(real_path):
                counters['missing'] += 1
                continue

            cap = cv2.VideoCapture(real_path)
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

            if total_frames <= 0:
                cap.release()
                counters['errors'] += 1
                continue

            frame_indices = np.linspace(0, total_frames - 1, frames_per_video, dtype=int)

            for i, frame_idx in enumerate(frame_indices):
                cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
                ret, frame = cap.read()
                if not ret: continue

                final_img = frame

                if crop_faces:
                    # --- MODIFICA 2: Conversione BGR -> RGB per MTCNN ---
                    # MTCNN funziona meglio su RGB e immagini PIL
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    pil_img = Image.fromarray(frame_rgb)

                    try:
                        # Detection
                        boxes, _ = self.mtcnn.detect(pil_img)

                        if boxes is not None:
                            # Prende il primo box (il più grande grazie a select_largest=True)
                            box = boxes[0]

                            # Coordinate MTCNN sono float: x1, y1, x2, y2
                            x1, y1, x2, y2 = [int(b) for b in box]

                            # Clip coordinate per evitare errori fuori immagine
                            x1 = max(0, x1)
                            y1 = max(0, y1)
                            x2 = min(frame.shape[1], x2)
                            y2 = min(frame.shape[0], y2)

                            # Eseguiamo il crop sul frame originale (BGR) così cv2.imwrite funziona
                            final_img = frame[y1:y2, x1:x2]

                            # Controllo di sicurezza se il crop è vuoto
                            if final_img.size == 0:
                                continue
                        else:
                            # Nessun volto trovato
                            continue

                    except Exception as e:
                        print(f"MTCNN Error su {real_path}: {e}")
                        continue

                try:
                    final_img = cv2.resize(final_img, target_size)

                    video_basename = os.path.basename(original_path).replace('/', '_').replace('.', '_')
                    filename = f"{video_basename}_f{i}.jpg"
                    save_path = os.path.join(self.output_dir, split, label, filename)

                    cv2.imwrite(save_path, final_img)

                except Exception as e:
                    counters['errors'] += 1
                    continue

            cap.release()
            counters['processed'] += 1

        print("\n--- Processo Completato ---")
        print(f"Video processati: {counters['processed']}")
        print(f"Video mancanti: {counters['missing']}")
        print(f"Errori lettura/resize: {counters['errors']}")

# --- ESEMPIO DI UTILIZZO ---
# 1. Carica metadati
metadata_df = pd.read_csv("/content/drive/MyDrive/anti_spoofing_dataset/dataset/metadata.csv")

# 2. Imposta path
VIDEOS_DIR = "/content/drive/MyDrive/anti_spoofing_dataset/dataset/videos/"
OUTPUT_DIR = "/content/drive/MyDrive/dataset_final"

# 3. Esegui
builder = LivenessDatasetBuilder(metadata_df, VIDEOS_DIR, OUTPUT_DIR)
builder.create_splits()
builder.extract_and_create(frames_per_video=15, crop_faces=True)

MTCNN running on device: cpu
Calcolo degli split Train/Val/Test sui video univoci...
Split completato. Train: 775 | Val: 222 | Test: 111
Inizio estrazione dataset in: /content/drive/MyDrive/dataset_final


  0%|          | 0/1150 [00:00<?, ?it/s]


--- Processo Completato ---
Video processati: 1138
Video mancanti: 12
Errori lettura/resize: 0


In [ ]:
import os
import torch
from torch.utils.data import Dataset
from PIL import Image

class SpoofingDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        """
        Args:
            root_dir (string): Percorso della cartella principale (che contiene train/val/test).
            split (string): Lo split da caricare ('train', 'val', o 'test').
            transform (callable, optional): Trasformazioni da applicare (es. Resize, ToTensor, MTCNN).
        """
        self.root_dir = root_dir
        self.split = split
        self.transform = transform

        # Definiamo il percorso base per questo split (es. ./dataset/train)
        self.split_dir = os.path.join(root_dir, split)

        # Definiamo le classi e il mapping
        # 0: Spoof (Attacco)
        # 1: Live (Reale)
        self.class_to_idx = {'spoof': 0, 'live': 1}
        self.samples = []

        # Verifica che la cartella dello split esista
        if not os.path.exists(self.split_dir):
            raise FileNotFoundError(f"La cartella dello split non esiste: {self.split_dir}")

        # Scansione delle cartelle 'live' e 'spoof'
        for class_name, label in self.class_to_idx.items():
            class_path = os.path.join(self.split_dir, class_name)

            if not os.path.exists(class_path):
                print(f"Attenzione: Cartella '{class_name}' non trovata in {self.split_dir}. Salto.")
                continue

            # Raccoglie tutte le immagini valide
            for filename in os.listdir(class_path):
                if filename.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.heic')):
                    full_path = os.path.join(class_path, filename)
                    self.samples.append((full_path, label))

        print(f"Caricato split '{split}': {len(self.samples)} immagini trovate.")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]

        try:
            # Carica immagine e converte in RGB (fondamentale per evitare errori con immagini RGBA o Grayscale)
            image = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Errore caricamento {img_path}: {e}")
            # Restituisce un tensore nero in caso di errore per non bloccare il training
            return torch.zeros(3, 160, 160), label

        # Applica le trasformazioni (es. MTCNN crop, Resize, Augmentation)
        if self.transform:
            image = self.transform(image)

        # Se la trasformazione non converte in tensore (es. MTCNN a volte ritorna PIL), fallo qui
        if not isinstance(image, torch.Tensor):
            from torchvision import transforms
            image = transforms.ToTensor()(image)

        return image, label